# RQ2: staged ApET and pure random — full MMStar1500

Run this after reviewing GQA. Same nine conditions as TextVQA/GQA: full576,
native96→32/native288→96, and three matched pure-random seeds at each schedule.
Full run: **13,500 primary predictions + 32 controls**. Every question is retained,
including baseline-wrong cases. Existing prompts and explicit-choice-v2 scoring.

Use **Internet + T4 ×2**. No input upload is needed; the pinned MMStar parquet
is downloaded automatically. Set `RUN_STAGED_MMSTAR = True`.
Keep `MMSTAR_SHARDS = [0, 1, 2, 3, 4, 5]` for all1500, or start with `[0]`
(250 questions), then `[1, 2, 3, 4, 5]` in a fresh output. The six fixed parts
keep repeated images together. Download each ZIP and matching log.
See `research/RQ2_STAGED_MMSTAR1500.md`. Do not run alongside another model job.


In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

RUN_STAGED_MMSTAR = False  # Set True: All1500 questions; optional six fixed parts.
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
MMSTAR_SHARDS = [0, 1, 2, 3, 4, 5]  # All1500; optional [0] first250, then [1, 2, 3, 4, 5].
INPUTS_DIR = None  # Optional existing prepared full MMStar1500 input directory.
CHECKPOINT_ARGS = []  # Optional existing original-format model and vision paths.

if not RUN_STAGED_MMSTAR:
    raise RuntimeError("Set RUN_STAGED_MMSTAR = True, then run cells manually.")
print("Preparing staged MMStar1500 code; model inference has not started.", flush=True)
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True, timeout=120)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True, timeout=120)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True, timeout=120)
if not (repo / "scripts/run_rq2_staged_mmstar1500.py").is_file():
    raise RuntimeError("Push the new staged MMStar1500 files first, then update this checkout.")
original_python = repo / "environments/original/.venv/bin/python"
rq2_staged_mmstar_runs = []
print("Repository:", repo, flush=True)


## Prepare the locked runtime and pinned MMStar inputs

The existing environment/checkpoint cache is reused in the same live session.
A fresh session restores the same locked original runtime. Preparation downloads
the hash-pinned parquet, checks all 1,500 questions/labels/images against the old
baseline identities, and decodes each image before model loading. A valid input
directory is reused. No RQ1 ZIP, Dataset attachment or historical results upload.


In [ ]:
setup_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
setup_log = Path(f"/kaggle/working/rq2-staged-mmstar-setup-{setup_stamp}.log")
runtime_report = Path(f"/kaggle/working/rq2-staged-mmstar-runtime-{setup_stamp}.json")

def stream_command(command, stream, *, cwd=None):
    print("Running:", " ".join(map(str, command)), flush=True)
    stream.write("COMMAND " + repr(command) + "\n")
    stream.flush()
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8", bufsize=1)
    lines = []
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
            lines.append(line)
        status = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    stream.write(f"EXIT {status}\n")
    stream.flush()
    if status:
        raise RuntimeError(f"Command failed (exit {status}); preserve {stream.name} before changing anything.")
    return "".join(lines)

with setup_log.open("x", encoding="utf-8") as stream:
    stream_command([
        "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
        "--output", str(runtime_report),
    ], stream, cwd=repo)
print("Locked original runtime ready. Run the prerequisite/helper cell next.", flush=True)
print("Setup log:", setup_log)
print("Runtime report:", runtime_report)

# Reuse verified inputs; otherwise download/prepare the same pinned MMStar parquet.
inputs = Path(INPUTS_DIR) if INPUTS_DIR is not None else Path("/kaggle/working/rq2-staged-mmstar1500-inputs")
with setup_log.open("a", encoding="utf-8") as stream:
    stream_command([str(original_python), "-X", "utf8", "-I", "scripts/prepare_rq2_staged_mmstar.py",
        "--output", str(inputs)], stream, cwd=repo)
print("Full MMStar inputs:", inputs, flush=True)


## Run current prerequisites

Actual tiny native tests cover stage16, RNG continuation, pure-random guards,
no-op replay and cache geometry. New CPU checks cover all1500 identities,
shared-image shards, scoring/format failures, export and shard combining.
All tests must pass with **zero skips** before pretrained weights load.


In [ ]:
def verify_rq2_staged_mmstar_prerequisites(*, refresh_runtime=False):
    globals()["rq2_staged_mmstar_prerequisites_passed"] = False
    with setup_log.open("a", encoding="utf-8") as stream:
        if refresh_runtime:
            stream_command([str(original_python), "-X", "utf8", "-I", "scripts/check_original_env.py",
                "--require-cuda", "--expected-gpus", "2", "--output", str(runtime_report)], stream, cwd=repo)
        output = stream_command([str(original_python), "-X", "utf8", "-m", "unittest",
            "research.tests.test_rq2_native", "research.tests.test_rq2_staged_protocol",
            "research.tests.test_rq2_staged_runner", "research.tests.test_rq2_staged_native",
            "research.tests.test_rq2_staged_mmstar_protocol", "research.tests.test_rq2_staged_mmstar_runner", "-v"], stream, cwd=repo)
        counts = re.findall(r"Ran (\d+) tests?", output)
        if not counts or int(counts[-1]) <= 26 or re.search(r"skipped\s*=", output):
            raise RuntimeError("Require all actual staged native tests and zero skips; preserve setup log.")
    command = [str(original_python), "-X", "utf8", "-I", "-c",
        "import json,sys; from pathlib import Path; sys.path.insert(0,str(Path.cwd()/'scripts')); "
        "from run_rq2_staged_mmstar1500 import prerequisite_signature,TESTS; "
        "print(json.dumps({'source_sha256':prerequisite_signature(),'test_modules':list(TESTS)}))"]
    signature = json.loads(subprocess.run(command, cwd=repo, capture_output=True, text=True,
                                          check=True, timeout=120).stdout)
    import hashlib
    proof = {"version": "rq2-staged-mmstar1500-v1", "status": "passed", "tests_run": int(counts[-1]),
        "skipped": 0, **signature,
        "runtime_report_sha256": hashlib.sha256(runtime_report.read_bytes()).hexdigest(),
        "setup_log_sha256": hashlib.sha256(setup_log.read_bytes()).hexdigest()}
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    globals()["staged_mmstar_prerequisite_report"] = Path(f"/kaggle/working/rq2-staged-mmstar-prerequisites-{stamp}.json")
    staged_mmstar_prerequisite_report.write_text(json.dumps(proof, indent=2) + "\n")
    globals()["rq2_staged_mmstar_prerequisites_passed"] = True
    print("Staged prerequisites passed:", proof["tests_run"], "tests, zero skips.", flush=True)


def run_rq2_staged_mmstar1500(shards=None):
    if not globals().get("rq2_staged_mmstar_prerequisites_passed", False):
        raise RuntimeError("Run verify_rq2_staged_mmstar_prerequisites() successfully before loading pretrained weights.")
    shards = MMSTAR_SHARDS if shards is None else shards
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    folder = Path(f"/kaggle/working/rq2-staged-mmstar1500-v1-{stamp}")
    archive, log = folder.with_suffix(".zip"), folder.with_suffix(".log")
    globals().setdefault("rq2_staged_mmstar_runs", []).append(folder)
    command = [str(original_python), "-X", "utf8", "-I", "scripts/run_rq2_staged_mmstar1500.py",
        "--inputs", str(inputs), "--output", str(folder), "--run-mmstar",
        "--shards", *map(str, shards),
        "--require-cuda", "--expected-gpus", "2", "--download-budget-gib", "20",
        "--setup-log", str(setup_log), "--runtime-report", str(runtime_report),
        "--prerequisite-report", str(staged_mmstar_prerequisite_report), *CHECKPOINT_ARGS]
    print("Run directory:", folder, flush=True)
    print("One model load across requested MMStar parts. Parts:", shards, flush=True)
    fallback_archive = False
    try:
        with log.open("x", encoding="utf-8") as stream:
            stream_command(command, stream, cwd=repo)
    finally:
        if not archive.exists():
            fallback_archive = True
            fallback = folder / "notebook-fallback"
            fallback.mkdir(parents=True, exist_ok=True)
            (fallback / "README.txt").write_text("Incomplete fallback: normal runner ZIP missing; no successful MMStar shard execution established.\n")
            for path in [setup_log, runtime_report, staged_mmstar_prerequisite_report, log]:
                if path.is_file():
                    shutil.copy2(path, fallback / path.name)
            with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as bundle:
                for path in sorted(folder.rglob("*")):
                    if path.is_file():
                        bundle.write(path, path.relative_to(folder))
        print("Download:", archive, "and", log, flush=True)
        if (folder / "report.json").is_file():
            report = json.loads((folder / "report.json").read_text())
            for key in ("status", "execution_passed", "progress", "wall_seconds", "error"):
                if key in report:
                    print(key + ":", report[key], flush=True)
    if fallback_archive:
        raise RuntimeError("Runner ZIP missing; preserve fallback ZIP/log for diagnosis.")
    report = json.loads((folder / "report.json").read_text())
    if report.get("version") != "rq2-staged-mmstar1500-v1" or report.get("status") != "passed" or not report.get("execution_passed"):
        raise RuntimeError("Staged MMStar execution did not pass; download ZIP/log for review.")
    summary = report["summaries"]
    print("Completed coverage:", summary["questions"], "questions;", summary["images"], "images; full1500:", summary["full_mmstar1500"], flush=True)
    for name, value in summary["conditions"].items():
        print(name, "accuracy=", round(value["accuracy"], 4), "APR=", value["APR"], flush=True)
    print("Requested shards passed. Download ZIP + log; keep disjoint exports for local combining.", flush=True)
    return folder


# Re-executing this cell invalidates the previous test pass.
globals()["rq2_staged_mmstar_prerequisites_passed"] = False


In [ ]:
verify_rq2_staged_mmstar_prerequisites()


## Run the requested fixed parts

All six parts: **1,500 questions / 13,500 primary predictions + 32 controls**.
For `[0]`: 250 questions / 2,250 primary predictions, plus that part's controls.
Use its observed runtime/ETA to plan the remaining parts. A part is not the full
benchmark. No new pretrained calibration is required; the calibrated adapter
and random streams are unchanged. All actual token counts/cache steps are checked.


In [ ]:
staged_mmstar_dir = run_rq2_staged_mmstar1500()


## Download ZIP and log

Keep **rq2-staged-mmstar1500-v1-&lt;timestamp&gt;.zip + matching .log** for each run.
The ZIP includes raw predictions, case/encoding records, original/model-canvas
images, source/runtime proof, CSV, category statistics and a local gallery.
Only all six parts support the full1500 comparison. Combine disjoint passed ZIPs
locally with `scripts/summarize_rq2_staged_mmstar1500.py`.


In [ ]:
from IPython.display import FileLink, display
paths = [setup_log, runtime_report, staged_mmstar_prerequisite_report]
for folder in rq2_staged_mmstar_runs:
    paths.extend([folder.with_suffix(".zip"), folder.with_suffix(".log")])
for path in paths:
    if path.exists():
        display(FileLink(str(path)))
